In [2]:
import sympy as sp
from IPython.display import display, Math

def analizar_fourier_complejo_master(f_t, variable_t, periodo_T, inicio_intervalo, m=3):
    """
    Analizador Completo Integrado de Series Complejas de Fourier.
    
    Genera:
    1. Coeficientes c_0 y c_n (fórmulas generales).
    2. Evaluación simbólica para n par (2k) y n impar (2k+1).
    3. Notación formal en sumatoria.
    4. Serie desarrollada en m términos (desplegada y simplificada).
    5. Tabla clasificada (DC, Par, Impar) con c_k, |c_k| y Fase.
    """
    t = variable_t
    n = sp.Symbol('n', integer=True)
    k_sym = sp.Symbol('k', integer=True)
    w0 = 2 * sp.pi / periodo_T
    fin_intervalo = inicio_intervalo + periodo_T
    
    # -------------------------------------------------------------
    # 1. CÁLCULO DE COEFICIENTES GENERALES
    # -------------------------------------------------------------
    # Coeficiente c0 (DC)
    c0 = sp.simplify((1 / periodo_T) * sp.integrate(f_t, (t, inicio_intervalo, fin_intervalo)))
    
    # Coeficiente cn (para n != 0)
    n_nz = sp.Symbol('n', integer=True, nonzero=True)
    cn_expr = (1 / periodo_T) * sp.integrate(f_t * sp.exp(-sp.I * n_nz * w0 * t), (t, inicio_intervalo, fin_intervalo))
    cn_expr = sp.simplify(cn_expr).subs(n_nz, n)
    
    # -------------------------------------------------------------
    # 2. ANÁLISIS SIMBÓLICO PAR / IMPAR
    # -------------------------------------------------------------
    cn_par = sp.simplify(cn_expr.subs(n, 2 * k_sym))
    cn_impar = sp.simplify(cn_expr.subs(n, 2 * k_sym + 1))
    
    # -------------------------------------------------------------
    # 3. EVALUACIÓN DE TÉRMINOS Y CONSTRUCCIÓN DE LA TABLA
    # -------------------------------------------------------------
    lista_terminos = []
    suma_evaluada = sp.S(0)
    tabla_coeficientes = []
    
    for val_k in range(-m, m + 1):
        # Identificar tipo de armónico
        if val_k == 0:
            tipo = "DC (n=0)"
            c_k = c0
        elif val_k % 2 == 0:
            tipo = "Par"
            c_k = cn_expr.subs(n, val_k)
        else:
            tipo = "Impar"
            c_k = cn_expr.subs(n, val_k)
            
        # Manejo de indeterminaciones (0/0)
        if c_k.has(sp.nan) or c_k.has(sp.zoo):
            c_k = sp.limit(cn_expr, n, val_k)
            
        c_k = sp.simplify(c_k)
        mag = sp.simplify(sp.Abs(c_k))
        fase = sp.simplify(sp.arg(c_k))
        
        # Guardar datos estructurados
        tabla_coeficientes.append({
            'n': val_k,
            'tipo': tipo,
            'c_k': c_k,
            'magnitud': mag,
            'fase': fase
        })
        
        # Acumular términos no nulos para la serie desplegada
        if c_k != 0:
            termino_k = c_k * sp.exp(sp.I * val_k * w0 * t)
            lista_terminos.append(termino_k)
            suma_evaluada += termino_k

    expresion_desarrollada = sp.Add(*lista_terminos, evaluate=False) if lista_terminos else 0

    # -------------------------------------------------------------
    # 4. SALIDA VISUAL EN JUPYTER (LATEX)
    # -------------------------------------------------------------
    print("=" * 75)
    print(" 1. COEFICIENTES GENERALES Y ANÁLISIS DE PARIDAD (n PAR / IMPAR)")
    print("=" * 75)
    display(Math(rf"c_0 = {sp.latex(c0)}"))
    display(Math(rf"c_n = {sp.latex(cn_expr)} \quad \text{{para }} n \neq 0"))
    display(Math(rf"\text{{Si }} n \text{{ es Par }} (n = 2k, k \neq 0): \quad c_{{2k}} = {sp.latex(cn_par)}"))
    display(Math(rf"\text{{Si }} n \text{{ es Impar }} (n = 2k + 1): \quad c_{{2k+1}} = {sp.latex(cn_impar)}"))
    
    print("\n" + "=" * 75)
    print(" 2. REPRESENTACIÓN FORMAL (NOTACIÓN SUMATORIA)")
    print("=" * 75)
    display(Math(rf"f(t) = \sum_{{n=-\infty}}^{{\infty}} \left( {sp.latex(cn_expr)} \right) e^{{i n \left({sp.latex(w0)}\right) t}}"))
    
    print("\n" + "=" * 75)
    print(f" 3. SERIE DESARROLLADA PARA m = {m} (-{m} <= n <= {m})")
    print("=" * 75)
    print("Términos explícitos sumados:")
    display(Math(rf"f_{{{m}}}(t) = {sp.latex(expresion_desarrollada)}"))
    print("Expresión reducida equivalente:")
    display(Math(rf"f_{{{m}}}(t) = {sp.latex(sp.simplify(suma_evaluada))}"))
    
    print("\n" + "=" * 75)
    print(f" 4. TABLA CLASIFICADA DE COEFICIENTES (PAR/IMPAR, MAGNITUD Y FASE)")
    print("=" * 75)
    for fila in tabla_coeficientes:
        v_n = fila['n']
        tipo = fila['tipo']
        ck_val = fila['c_k']
        mag = fila['magnitud']
        fase = fila['fase']
        display(Math(
            rf"n = {v_n:3d} \;\; \left[\text{{{tipo}}}\right]: \quad "
            rf"c_{{{v_n}}} = {sp.latex(ck_val)} \quad \implies \quad "
            rf"|c_{{{v_n}}}| = {sp.latex(mag)}, \quad "
            rf"\angle c_{{{v_n}}} = {sp.latex(fase)}"
        ))

    return {
        'c0': c0,
        'cn': cn_expr,
        'cn_par': cn_par,
        'cn_impar': cn_impar,
        'serie_desarrollada': expresion_desarrollada,
        'serie_reducida': sp.simplify(suma_evaluada),
        'tabla': tabla_coeficientes
    }

In [3]:
t = sp.Symbol('t', real=True)

# Ejemplo: Onda cuadrada desfasada
# f(t) = -1 en [-1, 0) y f(t) = 1 en [0, 1] (Periodo T = 2)
T = 4
f_t = sp.Piecewise(
    (1, (t >= -1) & (t <= 1)),
    (0, (t  > 1)&( t <= 3))
)

# Ejecutar el análisis completo para m = 4 armónicos
resultado = analizar_fourier_complejo_master(
    f_t=f_t, 
    variable_t=t, 
    periodo_T=T, 
    inicio_intervalo=-1, #mira el incio del intervalo
    m=4
)

 1. COEFICIENTES GENERALES Y ANÁLISIS DE PARIDAD (n PAR / IMPAR)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
t = sp.Symbol('t', real=True)
A = sp.Symbol('A', real=True, positive=True)
w = sp.Symbol('w', real=True, positive=True)

# Ejemplo: Onda triangular o cuadrada
T = 2 * sp.pi / w   
f_t = sp.Piecewise(
    (A * sp.sin(w * t), (t >= 0) & (t < sp.pi / w)),
    (0,                 (t >= sp.pi / w) & (t <= 2 * sp.pi / w))
)

# Corre todo de una vez para m = 3 armónicos
resultado = analizar_fourier_complejo_master(f_t, variable_t=t, periodo_T=T, inicio_intervalo=0, m=3)

 1. COEFICIENTES GENERALES Y ANÁLISIS DE PARIDAD (n PAR / IMPAR)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>